# Apertus StressLab — Colab smoke test (free tier)

Runs one **real** inference with the official `swiss-ai/Apertus-v1.5-8B` checkpoint and stores it in a timestamped run directory.

**Before you start (one-time, free):**
1. Create a free Hugging Face account, open https://huggingface.co/swiss-ai/Apertus-v1.5-8B and accept the license / AUP (the repo is gated).
2. Create a *read* token at https://huggingface.co/settings/tokens.
3. In Colab: key icon (Secrets) in the left sidebar → add a secret named `HF_TOKEN` with the token, and enable notebook access. Never paste the token into a cell.
4. `Runtime → Change runtime type → T4 GPU`.

**Why 8-bit:** the checkpoint is ~9B parameters (text + vision + audio). In bf16 that is ~18 GB of weights, which does not fit on the T4's 15 GB, and the T4 has no native bf16. We load it with bitsandbytes 8-bit (≈10 GB). The quantization mode is recorded in `metadata.json` — results from a quantized model must be reported as such.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Option A (current, repo not public yet): on your PC run `python scripts/make_bundle.py`, then upload `dist/apertus-stresslab-bundle.zip` with the cell below.

Option B (once the repo is public): replace this cell with `!git clone <repo-url> /content/apertus-stresslab`.

In [ ]:
import os, zipfile
from google.colab import files

uploaded = files.upload()  # choose apertus-stresslab-bundle.zip
bundle = next(iter(uploaded))
with zipfile.ZipFile(bundle) as zf:
    zf.extractall("/content")
os.chdir("/content/apertus-stresslab")
print(os.getcwd())

## 2. Install dependencies (Colab already has a CUDA build of torch)

In [ ]:
%cd /content/apertus-stresslab
!pip install -q -r requirements-inference.txt
!pip install -q -e .

## 3. Hugging Face token from Colab Secrets (never printed, never stored in results)

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
!python -m stresslab env

## 4. Phase 0: one real prompt
First run downloads ~18 GB of weights (several minutes). Greedy decoding, seed 42.

In [ ]:
!python -m stresslab infer --quantization 8bit --enable-thinking false \
    --prompt "Context: Maria Keller was born in Bern in 1981. Question: Where and when was Maria Keller born? Answer using only the context."

## 5. Smoke suite (4 cases)

In [ ]:
!python -m stresslab run --suite smoke --quantization 8bit --enable-thinking false

## 6. Download the results (Colab storage is wiped when the runtime stops)

In [ ]:
!ls -R results
!cd /content/apertus-stresslab && zip -qr /content/stresslab-results.zip results
from google.colab import files
files.download("/content/stresslab-results.zip")

Unzip into `C:\Extra\apertus-stresslab\results\` on your PC. Then `python -m stresslab show results\<run_id>`.

If 8-bit loading fails for the vision/audio towers, retry with `--quantization 4bit` and note it; do not switch to a different model silently.